<a href="https://colab.research.google.com/github/sahanabytes/machine-learning-portfolio/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sahanabytes/machine-learning-portfolio/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I selected five numerical features from the March 2026 reporting data to describe each webpage's search performance. These include Google Search impressions, clicks, average position, and Google Analytics pageviews and sessions. I will check missing values and feature availability before preparing the data. April 2026 performance will be kept separate as the future outcome, not used as an input feature.

In [8]:
!pip -q install duckdb huggingface_hub

import duckdb
from huggingface_hub import login, get_token

login()

hf_token = get_token()
con = duckdb.connect()

con.execute("DROP SECRET IF EXISTS hf_token")

con.execute("""
    CREATE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN '""" + hf_token + """'
    )
""")

print("DuckDB authentication configured")

DuckDB authentication configured


In [9]:
# Build the March 2026 feature vector

import pandas as pd

march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
)

feature_vector = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_pageviews) AS ga4_pageviews,
        SUM(ga4_sessions) AS ga4_sessions,
        MAX(CAST(gsc_data_available AS INTEGER)) AS gsc_data_available,
        MAX(CAST(ga4_data_available AS INTEGER)) AS ga4_data_available
    FROM read_parquet('{march_path}')
    GROUP BY client_hash_id, content_hash_id
""").df()

# Preserve missing-value information before filling values
numeric_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]

for col in numeric_features:
    feature_vector[col + "_was_missing"] = (
        feature_vector[col].isna().astype(int)
    )

    median_value = feature_vector[col].median()
    feature_vector[col] = feature_vector[col].fillna(
        median_value if pd.notna(median_value) else 0
    )

# Availability flags are categorical/binary features
feature_vector["gsc_data_available"] = (
    feature_vector["gsc_data_available"].fillna(0).astype(int)
)

feature_vector["ga4_data_available"] = (
    feature_vector["ga4_data_available"].fillna(0).astype(int)
)

print("Feature vector shape:", feature_vector.shape)
print("\nFeature vector preview:")
print(feature_vector.head())

print("\nRemaining missing values:")
print(feature_vector.isna().sum().sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector shape: (331437, 14)

Feature vector preview:
            client_hash_id           content_hash_id  gsc_impressions  \
0  client_62f4a7e64f5e0096  content_d0dff76c889de68f            181.0   
1  client_62f4a7e64f5e0096  content_67741cce996cfafa             46.0   
2  client_62f4a7e64f5e0096  content_2e6360ad20fd7107            899.0   
3  client_62f4a7e64f5e0096  content_ac8663da7484669a             34.0   
4  client_62f4a7e64f5e0096  content_65c50dfe9d87a585           3108.0   

   gsc_clicks  gsc_avg_position  ga4_pageviews  ga4_sessions  \
0         0.0          5.147402            0.0           0.0   
1         1.0          4.828125            0.0           0.0   
2         1.0          5.145765            0.0           0.0   
3         0.0          4.909314            0.0           0.0   
4         0.0          6.969536            0.0           0.0   

   gsc_data_available  ga4_data_available  gsc_impressions_was_missing  \
0                   1                   0 

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

I selected five features for the model:

1. `gsc_impressions`: Number of times a webpage appeared in Google Search results.
2. `gsc_clicks`: Number of clicks received from Google Search.
3. `gsc_avg_position`: Average ranking position of the webpage in search results.
4. `ga4_pageviews`: Number of times the webpage was viewed.
5. `ga4_sessions`: Number of sessions involving the webpage.

Missing numerical values will be handled carefully using median imputation where appropriate. Categorical values will be encoded before model training. These features are available from the March 2026 reporting data and are intended to represent information available before prediction. Their timing and availability must be verified to prevent data leakage.

In [10]:
# Feature notes: inspect the actual feature vector

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]

print("Available features:")
print([col for col in feature_columns if col in feature_vector.columns])

print("\nMissing values:")
print(feature_vector[feature_columns].isnull().sum())

print("\nFeature data types:")
print(feature_vector[feature_columns].dtypes)

print("\nAvailability flags:")
print(feature_vector[
    ["gsc_data_available", "ga4_data_available"]
].sum())

Available features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions']

Missing values:
gsc_impressions     0
gsc_clicks          0
gsc_avg_position    0
ga4_pageviews       0
ga4_sessions        0
dtype: int64

Feature data types:
gsc_impressions     float64
gsc_clicks          float64
gsc_avg_position    float64
ga4_pageviews       float64
ga4_sessions        float64
dtype: object

Availability flags:
gsc_data_available    176738
ga4_data_available     90489
dtype: int64


I used five numerical features: GSC impressions, GSC clicks, GSC average position, GA4 pageviews and GA4 sessions. These features describe search visibility, engagement and website traffic using March 2026 data.

Missing numerical values were filled using the median of each feature, and separate missing-value indicators were created to preserve information about missing data. GSC and GA4 availability flags were included as binary features. Client and content hash IDs are retained for grouping and identification, not as model inputs.

These features are based on March reporting data and are intended to be available before predicting April performance. However, this timing must be respected when training and evaluating the model.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I checked for data leakage by separating March 2026 input features from the April 2026 outcome. The target label is whether a content item received more than zero GSC clicks in April. April performance must not be included in the March feature vector. I also checked for label-related, future-period and product-related fields that could give the model information unavailable at prediction time.

In [11]:
# Leakage hunt: keep March features separate from April outcomes

april_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-04/data_0.parquet"
)

# Build the April outcome separately
april_outcomes = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS april_gsc_clicks,
        MAX(CAST(gsc_data_available AS INTEGER))
            AS april_gsc_data_available
    FROM read_parquet('{april_path}')
    GROUP BY client_hash_id, content_hash_id
""").df()

# Match March features with April outcomes
leakage_check = feature_vector.merge(
    april_outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Keep records with GSC data available in both periods
leakage_check = leakage_check[
    (leakage_check["gsc_data_available"] == 1) &
    (leakage_check["april_gsc_data_available"] == 1)
].copy()

# Create the future label separately from the features
leakage_check["april_positive_label"] = (
    leakage_check["april_gsc_clicks"] > 0
).astype(int)

# Define permitted March model inputs
model_input_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "gsc_data_available",
    "ga4_data_available",
    "gsc_impressions_was_missing",
    "gsc_clicks_was_missing",
    "gsc_avg_position_was_missing",
    "ga4_pageviews_was_missing",
    "ga4_sessions_was_missing"
]

# Check that future or label fields are not model inputs
forbidden_terms = [
    "april", "future", "label", "target", "outcome", "product"
]

leaked_columns = [
    col for col in model_input_columns
    if any(term in col.lower() for term in forbidden_terms)
]

print("Matched records:", len(leakage_check))
print(
    "April positive rate:",
    round(leakage_check["april_positive_label"].mean(), 4)
)
print("Future/label columns in model inputs:", leaked_columns)
print(
    "April outcome kept separate:",
    "april_positive_label" not in model_input_columns
)

assert len(leaked_columns) == 0
assert "april_positive_label" not in model_input_columns

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Matched records: 158549
April positive rate: 0.3735
Future/label columns in model inputs: []
April outcome kept separate: True


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

I excluded client_hash_id and content_hash_id from model inputs because they are identifiers, not meaningful performance features, and could create privacy and memorization risks. I excluded April GSC clicks and the April outcome label because they represent future information. I also excluded any product flags or label-derived fields because they could introduce leakage if they are not available at prediction time. Only March performance features and their availability and missingness indicators are retained as model inputs.

In [12]:
# Verify excluded fields and final model inputs

excluded_fields = {
    "client_hash_id": "Identifier; excluded to reduce privacy and memorization risks.",
    "content_hash_id": "Identifier; excluded to reduce privacy and memorization risks.",
    "april_gsc_clicks": "Future-period information; not available in March.",
    "april_positive_label": "Prediction target; must never be an input feature.",
    "product_flags": "Excluded because their availability and meaning are not verified."
}

print("Excluded fields and reasons:")
for field, reason in excluded_fields.items():
    print(f"- {field}: {reason}")

print("\nFinal model input columns:")
print(model_input_columns)

print("\nIdentifier columns excluded:")
print(
    "client_hash_id" not in model_input_columns
    and "content_hash_id" not in model_input_columns
)

print("\nApril outcome excluded:")
print(
    "april_positive_label" not in model_input_columns
    and "april_gsc_clicks" not in model_input_columns
)

print("\nFinal model input shape:", feature_vector[model_input_columns].shape)


Excluded fields and reasons:
- client_hash_id: Identifier; excluded to reduce privacy and memorization risks.
- content_hash_id: Identifier; excluded to reduce privacy and memorization risks.
- april_gsc_clicks: Future-period information; not available in March.
- april_positive_label: Prediction target; must never be an input feature.
- product_flags: Excluded because their availability and meaning are not verified.

Final model input columns:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions_was_missing', 'gsc_clicks_was_missing', 'gsc_avg_position_was_missing', 'ga4_pageviews_was_missing', 'ga4_sessions_was_missing']

Identifier columns excluded:
True

April outcome excluded:
True

Final model input shape: (331437, 12)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.